## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Anastacia Oliynyk

**Date:** 10/30/26

In [20]:
# Your Phase 1 solution to the problem goes here.

Q1:

1. What is the difference between regression and classification?

Regression is used to predict a numeric outcome, like baseline sales or price. Classification is used to predict a category or class, like vehicle class. In kNN, regression averages the outcomes of the k nearest neighbors, and classification takes their most common class.

2. What is a confusion table? What does it help us understand about a model's performance?

A confusion table, or confusion matrix, counts predictions by the actual class and the predicted class. The diagonal cells show correct predictions and the off diagonal cells are misclassifications. It helps us understand not only how many predictions were correct, but also which classes the model is confusing with one another, which accuracy alone doesn't show.

3. What does the SSE quantify about a particular model?

SSE, or sum of squared error, quantifies the model's prediction error by adding together the squared residuals for the validation observations. A residual is the difference between the observed and predicted value. A lower validation SSE means smaller prediction errors on the validation set.

4. What are overfitting and underfitting?

Overfitting occurs when a model fits the training data too closely, including noise in that data, and therefore may not generalize well to new observations. In kNN, a very small (k) can cause overfitting. Underfitting occurs when the model is too general and does not capture differences among observations. A very large k does this because predictions become nearly the same for every observation.

5. Why does splitting the data into training and testing sets, and choosing  𝑘  by evaluating accuracy or SSE on the test set, improve model performance?

Generalization is a model's ability to predict well on data not used to fit it. A model can fit its training data well by fitting noise, so training alone is misleading. Choosing k by training error alone favors a smaller k and leads to overfitting. Splitting lets us fit models on the training set and compare candidate values of k on a separate validation set that was not used to fit the model. We choose the k with the lowest validation SSE or highest validation accuracy.

6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

The class label is the most common class among the k neighbors. The probability distribution is each class's neighbor count divided by k. A strength of a class label is that it is a single, simple answer and accuracy is easy to compute from it. A weakness is that the class label hides the neighbor proportions, so a class may receive some support from the nearest neighbors without ever being the most common class. A strength of probability distribution is that it shows the proportion of neighbors in each class, so you keep information about uncertainty and can see that a class is a plausible runner-up even when it doesn't win. A weakness is that it isn't a single answer, so you need another step to turn it into a label.


In [21]:
#Q2

#Load the data. Perform some EDA, summarizing the target label and the features.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("land_mines.csv")
print(df.head())
print(df.shape)
print("\nMine type counts:")
print(df["mine_type"].value_counts().sort_index())

print("\nFeature summary:")
print(df[["voltage", "height", "soil"]].describe())

    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1
(338, 4)

Mine type counts:
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

Feature summary:
          voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000


the dataset contains 338 observations. the target variable is mine_type, which takes values from 1-5. the classes are fairly evenly represented, with 71 observations of mine type 1, 70 of type 2, 66 of each types 3 and 4, and 65 of type 5. the three predictor features are voltage, height, and soil. voltage ranges from 0.198 to 1, while height and soil both range from 0 to 1.

In [22]:
#Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)

X_raw = df[["voltage", "height", "soil"]]
y = df["mine_type"]

(
    X_train_raw,
    X_valid_raw,
    y_train,
    y_valid,
) = train_test_split(
    X_raw,
    y,
    test_size=0.50,
    random_state=65
)

print("Training class counts:")
print(y_train.value_counts().sort_index())

print("\nValidation class counts:")
print(y_valid.value_counts().sort_index())

Training class counts:
mine_type
1    37
2    32
3    36
4    28
5    36
Name: count, dtype: int64

Validation class counts:
mine_type
1    34
2    38
3    30
4    38
5    29
Name: count, dtype: int64


In [23]:
#Build a  𝑘 -NN classifier. Explain how you select  𝑘 .

ks = np.arange(1, 51)

valid_accuracy = []
train_accuracy = []

for k in ks:
    candidate_model = KNeighborsClassifier(n_neighbors=int(k))
    candidate_model.fit(X_train_scaled, y_train)

    valid_hat = candidate_model.predict(X_valid_scaled)
    train_hat = candidate_model.predict(X_train_scaled)

    valid_accuracy.append(
        np.mean(valid_hat == y_valid.to_numpy())
    )

    train_accuracy.append(
        np.mean(train_hat == y_train.to_numpy())
    )

k_star = int(ks[np.argmax(valid_accuracy)])

print("Optimal k:", k_star)
print("Validation accuracy:", max(valid_accuracy))

Optimal k: 2
Validation accuracy: 0.44970414201183434


I selected k by fitting k-NN classifiers for values of k from 1 through 50 on the training data. I then used each model to predict mine type in the validation set and compared their validation accuracy. the highest validation accuracy occurred at k=2, with an accuracy of about 0.450. therefore, i selected k=2.

In [24]:
#Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?

optimal_model = KNeighborsClassifier(n_neighbors=k_star)
optimal_model.fit(X_train_scaled, y_train)

valid_hat = optimal_model.predict(X_valid_scaled)

accuracy = np.mean(valid_hat == y_valid.to_numpy())

print("Optimal k:", k_star)
print("Accuracy:", accuracy)

labels = sorted(y.unique())

confusion_table = pd.crosstab(
    pd.Series(y_valid.to_numpy(), name="Actual"),
    pd.Series(valid_hat, name="Predicted")
)

confusion_table = confusion_table.reindex(
    index=labels,
    columns=labels,
    fill_value=0
)

print(confusion_table)

Optimal k: 2
Accuracy: 0.44970414201183434
Predicted   1   2   3  4  5
Actual                     
1          21   1   6  4  2
2           0  33   2  3  0
3           5   5  10  6  4
4          13   5  10  9  1
5          10   1   8  7  3


the optimal model uses k=2 and has a validation accuracy of about 44.97%. performance differs substantially by mine type. mine type 2 is classified most accurately with 33 of 38 observations classified correctly, or about 61.76%. performance is lower for mine types 3, 4, and 5. the model correctly identifies 10 of 30 type 3 mines (33.33%), 9 of 38 type 4 mines (23.68%), and only 3 of 29 type 5 mines (10.34%). the confusion table shows frequent confusion of types 4 and 5 with type 1

Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

I would use this model as a supporting tool rather than relying only on its predicted mine type. the confusion table shows that the model performs much better for some mine types than others and makes many errors for types 3,4, and especially 5. I would look at both predicted class and the proportions of the nearest neighbors belonging to each class. if the neighbors are divided across several mine types, the prediction should be treated with more uncertainty and should be verified additionally before being used in practice. this is important because incorrectly identifying a land mine could have very serious consequences.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:**
f2e608c
- **AI tool and model used:** ChatGPT GPT-5.6 Sol

### *Prompts used

1. Review my Phase 1 solution and tell me what I should improve for Phase 2. Focus on any errors, unclear explanations, unnecessary code, or parts that could be more complete, but keep my original approach and work as much as possible.

2. Go through each section of my Phase 1 solution and tell me exactly what I should change or clarify. Provide a concise numbered list. For anything that needs improvement, explain why and suggest a specific revision.

### *AI-proposed changes


1. Q1: Keep the answers mostly as written. They are clear and complete. Make only small wording/grammar fixes, such as changing “off diagonal” to “off-diagonal” and “a very small (k)” to “a very small k.” No major content changes are needed.
2. Q2 EDA: Keep the code and interpretation, but fix the writing. Change “66 of each types 3 and 4” to “66 each of types 3 and 4,” and capitalize the beginnings of sentences. The EDA itself already summarizes the sample size, target distribution, and ranges of the three features.
3. Add the missing feature-scaling step after the 50/50 split. The later model code uses X_train_scaled and X_valid_scaled, but those variables are never created in the Phase 1 notebook. Add a MinMaxScaler, fit it on X_train_raw, and use it to transform both the training and validation features before fitting the classifier. Otherwise, the notebook will not run correctly from a fresh session.
4. After the split, add one sentence interpreting the class counts. Both the training and validation sets contain observations from all five mine types, so say that explicitly. This helps show that the 50/50 split successfully avoided leaving an entire class out of either set.
5. Simplify the \(k\)-selection code or explain the extra training calculations. train_accuracy and train_hat are calculated but never used in the explanation or model selection. Either remove them to make the code cleaner, or keep them only if you plan to compare training and validation performance. Also capitalize “The” and “Therefore” in the written explanation.
6. Correct the mine type 2 percentage in the confusion-table interpretation. You wrote that 33 of 38 correctly classified type 2 mines equals 61.76%, but it is actually 86.84%. The 61.76% value belongs to mine type 1, where 21 of 34 are correct. Revise this section to say that type 2 performs best at 86.84%, followed by type 1 at 61.76%, while types 3, 4, and especially 5 perform much worse.
7. Keep the practical-use conclusion, but make the wording slightly clearer. Change “should be verified additionally” to “should receive additional verification.” Also explicitly mention that predictions involving mine type 5 deserve particular caution because that class had the lowest correct-classification rate. The rest of your point—that the model should be a supporting tool and that neighbor proportions can help communicate uncertainty—is worth keeping.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept | The Q1 answers were already correct, so I only made the suggested grammar and wording fixes. These changes improve readability without changing my original explanations. |
| 2 | Accept | My EDA included the correct information, but there were a few grammatical and capitalization errors. I corrected these while keeping the same results and interpretation. |
| 3 | Accept | The scaling step was missing even though X_train_scaled and X_valid_scaled were used later in my code. I added MinMaxScaler, fit it to the training data, and transformed both the training and validation features so the code runs correctly. |
| 4 | Accept | I added a sentence explaining that all five mine types appear in both the training and validation sets. I verified this using the class counts printed after the split. |
| 5 | Accept | I removed the unused training-accuracy calculations because I selected \(k\) using validation accuracy and did not use the training accuracy anywhere else. This makes the code simpler without changing the result. |
| 6 | Accept | I corrected the percentage for mine type 2. The confusion table shows 33 out of 38 type 2 mines were classified correctly, which is 86.84%. The 61.76% value actually corresponds to type 1, with 21 out of 34 correct. |
| 7 | Accept | I kept my original conclusion but clarified the wording and added that mine type 5 predictions should be treated with extra caution because it had the lowest correct-classification rate.

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [25]:
# Your Phase 2 solution to the problem goes here.

Q1
1. What is the difference between regression and classification?

Regression is used to predict a numeric outcome, like baseline sales or price. Classification is used to predict a category or class, like vehicle class. In kNN, regression averages the outcomes of the k nearest neighbors, and classification takes their most common class.

2. What is a confusion table? What does it help us understand about a model's performance?

A confusion table, or confusion matrix, counts predictions by the actual class and the predicted class. The diagonal cells show correct predictions and the off-diagonal cells are misclassifications. It helps us understand not only how many predictions were correct, but also which classes the model is confusing with one another, which accuracy alone doesn't show.

3. What does the SSE quantify about a particular model?

SSE, or sum of squared error, quantifies the model's prediction error by adding together the squared residuals for the validation observations. A residual is the difference between the observed and predicted value. A lower validation SSE means smaller prediction errors on the validation set.

4. What are overfitting and underfitting?

Overfitting occurs when a model fits the training data too closely, including noise in that data, and therefore may not generalize well to new observations. In kNN, a very small k can cause overfitting. Underfitting occurs when the model is too general and does not capture differences among observations. A very large k does this because predictions become nearly the same for every observation.

5. Why does splitting the data into training and testing sets, and choosing k by evaluating accuracy or SSE on the test set, improve model performance?

Generalization is a model's ability to predict well on data not used to fit it. A model can fit its training data well by fitting noise, so training alone is misleading. Choosing k by training error alone favors a smaller k and can lead to overfitting. Splitting lets us fit models on the training set and compare candidate values of k on a separate validation set that was not used to fit the model. We choose the k with the lowest validation SSE or highest validation accuracy.

6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

The class label is the most common class among the k neighbors. The probability distribution is each class's neighbor count divided by k. A strength of a class label is that it is a single, simple answer and accuracy is easy to compute from it. A weakness is that the class label hides the neighbor proportions, so a class may receive some support from the nearest neighbors without ever being the most common class. A strength of a probability distribution is that it shows the proportion of neighbors in each class, so you keep information about uncertainty and can see that a class is a plausible runner-up even when it doesn't win. A weakness is that it isn't a single answer, so you need another step to turn it into a label.

In [26]:
# Load the data and perform EDA

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("land_mines.csv")

print(df.head())
print(df.shape)

print("\nMine type counts:")
print(df["mine_type"].value_counts().sort_index())

print("\nFeature summary:")
print(df[["voltage", "height", "soil"]].describe())

    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1
(338, 4)

Mine type counts:
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

Feature summary:
          voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000


The dataset contains 338 observations. The target variable is mine_type, which takes values from 1 to 5. The classes are fairly evenly represented, with 71 observations of mine type 1, 70 of type 2, 66 each of types 3 and 4, and 65 of type 5. The three predictor features are voltage, height, and soil. Voltage ranges from about 0.198 to 1, while height and soil both range from 0 to 1.

In [27]:
# Split the sample 50/50 into training and validation sets

X_raw = df[["voltage", "height", "soil"]]
y = df["mine_type"]

(
    X_train_raw,
    X_valid_raw,
    y_train,
    y_valid,
) = train_test_split(
    X_raw,
    y,
    test_size=0.50,
    random_state=65
)

print("Training class counts:")
print(y_train.value_counts().sort_index())

print("\nValidation class counts:")
print(y_valid.value_counts().sort_index())

Training class counts:
mine_type
1    37
2    32
3    36
4    28
5    36
Name: count, dtype: int64

Validation class counts:
mine_type
1    34
2    38
3    30
4    38
5    29
Name: count, dtype: int64


In [28]:
# Scale the predictor features

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_raw)
X_valid_scaled = scaler.transform(X_valid_raw)

In [29]:
# Build a k-NN classifier and select k

ks = np.arange(1, 51)

valid_accuracy = []

for k in ks:
    candidate_model = KNeighborsClassifier(n_neighbors=int(k))
    candidate_model.fit(X_train_scaled, y_train)

    valid_hat = candidate_model.predict(X_valid_scaled)

    valid_accuracy.append(
        np.mean(valid_hat == y_valid.to_numpy())
    )

k_star = int(ks[np.argmax(valid_accuracy)])

print("Optimal k:", k_star)
print("Validation accuracy:", max(valid_accuracy))

Optimal k: 2
Validation accuracy: 0.44970414201183434


I selected k by fitting k-NN classifiers for values of k from 1 through 50 on the training data. I then used each model to predict mine type in the validation set and compared their validation accuracy. The highest validation accuracy occurred at k=2, with an accuracy of about 0.450. Therefore, I selected k=2.

In [30]:
# Print a confusion table for the optimal model

optimal_model = KNeighborsClassifier(n_neighbors=k_star)
optimal_model.fit(X_train_scaled, y_train)

valid_hat = optimal_model.predict(X_valid_scaled)

accuracy = np.mean(valid_hat == y_valid.to_numpy())

print("Optimal k:", k_star)
print("Accuracy:", accuracy)

labels = sorted(y.unique())

confusion_table = pd.crosstab(
    pd.Series(y_valid.to_numpy(), name="Actual"),
    pd.Series(valid_hat, name="Predicted")
)

confusion_table = confusion_table.reindex(
    index=labels,
    columns=labels,
    fill_value=0
)

print(confusion_table)

Optimal k: 2
Accuracy: 0.44970414201183434
Predicted   1   2   3  4  5
Actual                     
1          21   1   6  4  2
2           0  33   2  3  0
3           5   5  10  6  4
4          13   5  10  9  1
5          10   1   8  7  3


The optimal model uses k=2 and has a validation accuracy of about 44.97%. Performance differs substantially by mine type. Mine type 2 is classified most accurately, with 33 of 38 observations classified correctly, or about 86.84%. Mine type 1 is also classified relatively accurately, with 21 of 34 observations correct, or about 61.76%. Performance is lower for mine types 3, 4, and 5. The model correctly identifies 10 of 30 type 3 mines (33.33%), 9 of 38 type 4 mines (23.68%), and only 3 of 29 type 5 mines (10.34%). The confusion table also shows frequent confusion of types 4 and 5 with type 1.

5. How would you advise someone to use this predictive model in practice?

I would use this model as a supporting tool rather than relying only on its predicted mine type. The confusion table shows that the model performs much better for some mine types than others and makes many errors for types 3, 4, and especially 5. I would look at both the predicted class and the proportions of the nearest neighbors belonging to each class. If the neighbors are divided across several mine types, the prediction should be treated with more uncertainty and should receive additional verification before being used in practice. Predictions involving mine type 5 should be treated especially cautiously because the model correctly classified only 3 of the 29 type 5 mines. This is important because incorrectly identifying a land mine could have very serious consequences.

### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

The main improvements I made in phase 2 were fixing errors in my code and making some of my explanations clearer. the biggest issue was that I had used X_train_scaled and X_valid_scaled later in my code without actually creating them so i added the missing MinMaxScaler step. I also removed some unused training accuracy code and fixed smaller grammar and capitalization issues throughout my responses. Another important correction was in my confusion table interpretation. I originally wrote that 33 out of 38 type 2 mines was 61.76% but after checking the calculation the correct value was 86.84%.
one limitation of ai suggestion was that i still had to verify them against the actual notebook outputs instead of assuming they were correct. i checked the class counts, confusion table, accuracy, and percentages directly from the code output and reran the notebook after. the final model still has an overall validation accuracy of only about 44.97% and it performs especially poorly for mine type 5. because of that, my main remaining concern is that the model would not be reliable enough to use by itself for a high risk decision.